# VLA Inference Notebook (LangSAM + Optional Depth)

Set image path + prompt in the config cell and run all.


In [ ]:
# If needed in a fresh runtime:
# %pip install -e ..

import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from lang_sam import LangSAM
from lang_sam.utils import draw_image


In [ ]:
# -----------------------------
# HARD-CODE YOUR INPUTS HERE
# -----------------------------
image_path = Path('../datasets/spectralwaste/20221011_01_084638.png')
prompt = 'waste pile'

# Model settings
sam_type = 'sam2.1_hiera_large'
gdino_model_id = 'IDEA-Research/grounding-dino-base'
box_threshold = 0.30
text_threshold = 0.25

# Optional depth settings (set depth_path=None to disable volume/mass)
depth_path = Path('../depth_results/20221011_01_084638_depth_raw.png')  # or None
depth_scale_factor = 10000.0
real_world_height_m = 1.0
real_world_width_m = 1.0
min_height_threshold_m = 0.005
ground_percentile = 99.9
target_ground_depth_m = None
density_kg_per_m3 = 180.0


In [ ]:
# Load image
image_pil = Image.open(image_path).convert('RGB')
image_np = np.array(image_pil)
print('Image:', image_path, image_np.shape)

# Build model
model = LangSAM(sam_type=sam_type, gdino_model_id=gdino_model_id)


In [ ]:
# Inference
res = model.predict([image_pil], [prompt], box_threshold=box_threshold, text_threshold=text_threshold)[0]
masks = res.get('masks', np.zeros((0, image_np.shape[0], image_np.shape[1]), dtype=bool))
boxes = res.get('boxes', np.zeros((0, 4), dtype=np.float32))
scores = res.get('scores', np.zeros((0,), dtype=np.float32))
labels = res.get('labels', [])

print('Detections:', len(masks))
if len(masks):
    union_mask = np.any(masks.astype(bool), axis=0)
    coverage = float(np.mean(union_mask) * 100.0)
    print(f'Mask coverage: {coverage:.2f}%')
    overlay = draw_image(image_np, masks, boxes, scores, labels).astype(np.uint8)
else:
    union_mask = np.zeros(image_np.shape[:2], dtype=bool)
    overlay = image_np.copy()

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(image_np); ax[0].set_title('Input'); ax[0].axis('off')
ax[1].imshow(union_mask, cmap='gray'); ax[1].set_title('Union Mask'); ax[1].axis('off')
ax[2].imshow(overlay); ax[2].set_title('Overlay'); ax[2].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Optional: depth -> volume/mass
def estimate_height_map(depth_map, target_ground_depth, ground_percentile):
    valid = depth_map > 0
    d = depth_map[valid]
    if d.size == 0:
        return np.zeros_like(depth_map), 0.0, 1.0
    p = max(0.0, min(100.0, ground_percentile))
    g_est = float(np.percentile(d, p)) if p < 100.0 else float(np.max(d))
    scale = 1.0
    if target_ground_depth is not None and target_ground_depth > 0 and g_est > 0:
        scale = float(target_ground_depth) / g_est
    g = g_est * scale
    h = g - (depth_map * scale)
    h[~valid] = 0.0
    return h, g, scale

def pixel_area_m2(image_hw, real_h, real_w):
    H, W = image_hw
    return (real_h * real_w) / float(H * W)

if depth_path is None:
    print('depth_path is None -> skipping volume/mass')
else:
    depth_map = np.array(Image.open(depth_path), dtype=np.float64) / depth_scale_factor
    if depth_map.shape != union_mask.shape:
        raise ValueError(f'Shape mismatch depth={depth_map.shape} mask={union_mask.shape}')

    height_map, ground_depth, depth_scale = estimate_height_map(depth_map, target_ground_depth_m, ground_percentile)
    obj_mask = union_mask & (height_map > min_height_threshold_m)
    obj_heights = height_map[obj_mask]

    if obj_heights.size == 0:
        print('No positive-height object region after thresholding.')
    else:
        area_px = pixel_area_m2(height_map.shape, real_world_height_m, real_world_width_m)
        volume_m3 = float(np.sum(obj_heights) * area_px)
        volume_l = volume_m3 * 1000.0
        mass_kg = volume_m3 * density_kg_per_m3

        print(f'Ground depth: {ground_depth:.4f} m (scale={depth_scale:.4f})')
        print(f'Mean object height: {np.mean(obj_heights)*1000:.2f} mm')
        print(f'Estimated volume: {volume_l:.4f} L')
        print(f'Estimated mass (@{density_kg_per_m3} kg/m^3): {mass_kg:.4f} kg')

        plt.figure(figsize=(6, 5))
        plt.imshow(np.where(obj_mask, height_map, 0.0), cmap='hot')
        plt.title('Masked Height Map (m)')
        plt.axis('off')
        plt.colorbar()
        plt.show()
